# Mississippi -- Title 83 (Insurance) -> `data/mississippi/ins_codes/*.md`

Mississippi's **insurance** statutes are **Title 83** (official: [code.ms.gov](https://code.ms.gov/)). On **Justia**, Title 83 uses **chapter** and often an extra **slug** folder (e.g. `general-provisions`) before **`section-83-...`** URLs.

Justia mixes **year-scoped** paths (`/codes/mississippi/{CODE_YEAR}/title-83/...`) and **canonical** paths (`/codes/mississippi/title-83/...`). This notebook normalizes paths for discovery so both stay in scope.

**Cloudflare** often blocks plain **`httpx`**; this notebook uses **`curl_cffi`** with **`impersonate="chrome120"`** (same pattern as **`ins_ipynb/minnesota.ipynb`**).

**Discovery:** BFS from **`/codes/mississippi/{CODE_YEAR}/title-83/`**, following paths whose canonical form lies under **`/codes/mississippi/title-83/`**; every **`section-83-...`** link is collected (~**4,000** sections).

**Download:** text from **`div.primary-content`**, with Justia boilerplate stripped. Files are **`MS_sec_<label>.md`** where **`<label>`** is the segment after **`section-`** (hyphens to underscores, e.g. `MS_sec_83_1_1.md`).

**Config:** **`CODE_YEAR`** seeds the starting index URL. **`MAX_SECTIONS`** caps downloads (**0** = all). **`MAX_DISCOVERY_PAGES`** caps index fetches (**0** = no cap). **`REUSE_DISCOVERED_URLS`** skips discovery when **`_mississippi_title83_section_urls.txt`** exists.

Run with the **project root** as cwd (same as **`python -m app.ingest`**).

Then run **`python -m app.ingest`** from the project root.


In [1]:
%pip install -q curl_cffi beautifulsoup4


You should consider upgrading via the '/Users/apps/Downloads/ZProjects/RAG/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import re
import time
from collections import deque
from pathlib import Path
from urllib.parse import urljoin, urlparse

from bs4 import BeautifulSoup
from curl_cffi import requests as curl_requests

BASE = "https://law.justia.com"
CODE_YEAR = "2024"
PATH_PREFIX = "/codes/mississippi/title-83"
TITLE_INDEX = f"{BASE}/codes/mississippi/{CODE_YEAR}/title-83/"

OUT_DIR = Path("data") / "mississippi" / "ins_codes"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CURL_IMPERSONATE = "chrome120"
REQUEST_DELAY_SEC = 0.12
TIMEOUT = 60.0

MAX_SECTIONS = 0
MAX_DISCOVERY_PAGES = 0

SKIP_EXISTING = True

DISCOVERED_LIST = OUT_DIR / "_mississippi_title83_section_urls.txt"
REUSE_DISCOVERED_URLS = True

section_label_re = re.compile(r"/section-([^/]+)/?$", re.I)


In [3]:
def curl_get(url: str) -> str:
    time.sleep(REQUEST_DELAY_SEC)
    r = curl_requests.get(url, impersonate=CURL_IMPERSONATE, timeout=TIMEOUT)
    r.raise_for_status()
    return r.text


def path_key(u: str) -> str:
    return urlparse(u).path.rstrip("/")


def canonical(p: str) -> str:
    """Map /codes/mississippi/YYYY/... to /codes/mississippi/... for scope checks."""
    return re.sub(r"^(/codes/mississippi)/\d{4}/", r"\1/", p)


def in_scope(p: str) -> bool:
    c = canonical(p)
    return c == PATH_PREFIX or c.startswith(PATH_PREFIX + "/")


def is_section_path(p: str) -> bool:
    m = section_label_re.search(p)
    if not m:
        return False
    return m.group(1).lower().startswith("83-")


def discover_section_urls() -> list[str]:
    """BFS Title 83 index pages; collect section-83-... URLs."""
    start = TITLE_INDEX
    seen: set[str] = set()
    in_q: set[str] = {path_key(start)}
    q: deque[str] = deque([start])
    sections: set[str] = set()
    fetches = 0

    while q:
        if MAX_DISCOVERY_PAGES and fetches >= MAX_DISCOVERY_PAGES:
            break
        url = q.popleft()
        pk = path_key(url)
        in_q.discard(pk)
        if pk in seen:
            continue
        if is_section_path(pk):
            continue
        if not in_scope(pk):
            continue
        seen.add(pk)
        html = curl_get(url)
        fetches += 1
        soup = BeautifulSoup(html, "html.parser")
        for a in soup.find_all("a", href=True):
            absu = urljoin(url, a["href"]).split("#")[0]
            p = path_key(absu)
            if not in_scope(p):
                continue
            if is_section_path(p):
                sections.add(BASE + p + "/")
            else:
                if p in seen or p in in_q:
                    continue
                in_q.add(p)
                q.append(BASE + p + "/")

    return sorted(sections, key=lambda u: label_sort_key(statute_label_from_url(u)))


def statute_label_from_url(url: str) -> str:
    pk = path_key(url)
    m = section_label_re.search(pk)
    if not m:
        raise ValueError(f"cannot parse section from {url!r}")
    return m.group(1)


def label_sort_key(label: str) -> tuple:
    out: list[tuple[int, int | str]] = []
    for part in label.split("-"):
        if part.isdigit():
            out.append((0, int(part)))
        else:
            out.append((1, part.lower()))
    return tuple(out)


def label_to_filename(label: str) -> str:
    safe = label.replace("-", "_")
    return f"MS_sec_{safe}.md"


def extract_primary_text(html: str) -> tuple[str, str]:
    soup = BeautifulSoup(html, "html.parser")
    title_el = soup.find("title")
    title_txt = title_el.get_text(strip=True) if title_el else ""
    pc = soup.select_one("div.primary-content")
    if pc:
        text = pc.get_text("\n", strip=True)
    else:
        main = soup.find("main") or soup.find("article")
        text = main.get_text("\n", strip=True) if main else soup.get_text("\n", strip=True)
    return title_txt, text


def strip_justia_boilerplate(text: str) -> str:
    drop_prefixes = (
        "Go to Previous Versions",
        "View All Versions",
        "Learn more",
        "This media-neutral citation",
    )
    drop_exact = {
        "of",
        "this Section",
        "Universal Citation:",
        "Next",
        "Previous",
    }
    lines = text.split("\n")
    out: list[str] = []
    skip_until_substantive = True
    for line in lines:
        s = line.strip()
        if not s:
            if not skip_until_substantive:
                out.append("")
            continue
        if s in drop_exact:
            continue
        if any(s.startswith(p) for p in drop_prefixes):
            continue
        if s.startswith("20") and "Mississippi Code" in s:
            continue
        if s.startswith("Disclaimer:") or s.startswith("These codes may not"):
            continue
        skip_until_substantive = False
        out.append(s)
    return "\n".join(out).strip()


def download_title83() -> dict[str, int]:
    if REUSE_DISCOVERED_URLS and DISCOVERED_LIST.exists() and DISCOVERED_LIST.stat().st_size > 50:
        raw = [ln.strip() for ln in DISCOVERED_LIST.read_text(encoding="utf-8").splitlines() if ln.strip()]
        all_urls = sorted(raw, key=lambda u: label_sort_key(statute_label_from_url(u)))
        print(f"Loaded {len(all_urls)} section URLs from {DISCOVERED_LIST.name} (skipped discovery)")
    else:
        found = discover_section_urls()
        print(f"Discovered {len(found)} section URLs under Title 83")
        all_urls = found
        DISCOVERED_LIST.write_text("\n".join(all_urls) + "\n", encoding="utf-8")

    todo = all_urls if not MAX_SECTIONS else all_urls[:MAX_SECTIONS]
    if MAX_SECTIONS:
        print(f"Limited downloads to first {len(todo)} sections (MAX_SECTIONS)")

    wrote = skipped = failed = 0
    for i, sec_url in enumerate(todo, 1):
        label = statute_label_from_url(sec_url)
        dest = OUT_DIR / label_to_filename(label)
        if SKIP_EXISTING and dest.exists() and dest.stat().st_size > 80:
            skipped += 1
        else:
            try:
                html = curl_get(sec_url)
                head_t, body_t = extract_primary_text(html)
                body_t = strip_justia_boilerplate(body_t)
                title = head_t.split("::", 1)[0].strip() if head_t else f"Mississippi Code {label}"
                md = (
                    f"# {title}\n\n"
                    f"**Mississippi Code -- Title 83 (Insurance)**\n\n"
                    f"**Source (Justia mirror):** {sec_url}\n\n"
                    f"**Verify on official site:** [code.ms.gov](https://code.ms.gov/)\n\n"
                    f"**Section (URL label):** {label}\n\n"
                    f"---\n\n"
                    f"{body_t}\n"
                )
                dest.write_text(md, encoding="utf-8")
                wrote += 1
            except Exception as e:
                print(f"FAIL {label}: {e}")
                failed += 1
        if i % 200 == 0:
            print(f"… {i}/{len(todo)} (wrote={wrote} skipped={skipped} failed={failed})")

    print(f"Done. wrote={wrote} skipped={skipped} failed={failed} → {OUT_DIR.resolve()}")
    return {"wrote": wrote, "skipped": skipped, "failed": failed}


download_title83()


Discovered 4040 section URLs under Title 83
… 200/4040 (wrote=69 skipped=131 failed=0)
… 400/4040 (wrote=139 skipped=261 failed=0)
… 600/4040 (wrote=207 skipped=393 failed=0)
… 800/4040 (wrote=300 skipped=500 failed=0)
… 1000/4040 (wrote=368 skipped=632 failed=0)
… 1200/4040 (wrote=441 skipped=759 failed=0)
… 1400/4040 (wrote=515 skipped=885 failed=0)
… 1600/4040 (wrote=584 skipped=1016 failed=0)
… 1800/4040 (wrote=652 skipped=1148 failed=0)
… 2000/4040 (wrote=720 skipped=1280 failed=0)
… 2200/4040 (wrote=787 skipped=1413 failed=0)
… 2400/4040 (wrote=854 skipped=1546 failed=0)
… 2600/4040 (wrote=921 skipped=1679 failed=0)
… 2800/4040 (wrote=987 skipped=1813 failed=0)
… 3000/4040 (wrote=1056 skipped=1944 failed=0)
… 3200/4040 (wrote=1123 skipped=2077 failed=0)
… 3400/4040 (wrote=1190 skipped=2210 failed=0)
… 3600/4040 (wrote=1257 skipped=2343 failed=0)
… 3800/4040 (wrote=1324 skipped=2476 failed=0)
… 4000/4040 (wrote=1395 skipped=2605 failed=0)
Done. wrote=1412 skipped=2628 failed=0 → /

{'wrote': 1412, 'skipped': 2628, 'failed': 0}

## Next step

`python -m app.ingest` from the project root.
